# Lost in the Museum - contrastive fine-tuning

Zero-shot DINOv2 embeddings reached 0.8285 on this task. This notebook instead
**trains** the backbone for the specific invariance the task needs, which is
possible because the queries are synthetic degradations of gallery images:
rotation with pale padding, perspective, crop, blur, downscale.

That means labelled pairs can be manufactured without any labels being given --
degrade any image and you have a (query, target) pair with known ground truth --
and the model can be taught directly that a degraded copy must retrieve its
original.

### Measured on a 4,000-image subset with ViT-B before scaling up

| Degradation strength | Zero-shot | Fine-tuned |
|---|---|---|
| 1.25 (calibrated to the real leaderboard) | 0.727 | **0.993** |
| 1.50 | 0.517 | **0.957** |
| 2.00 (twice the training strength) | 0.197 | **0.680** |

Holding up at twice the training strength indicates genuine invariance rather
than memorisation of one degradation level.

### Why training on the evaluation images is legitimate here

All 20,000 images are provided and the split between gallery, query and
distractor is never revealed. The task is transductive: no label is used, only
the pixel content that every entrant is given.

**Settings:** GPU ON, Internet ON. Roughly 1.5-2 hours.


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None


def find_data_dir():
    """Locate the image folder without hard-coding a mount slug."""
    best, best_n = None, 0
    for d in Path("/kaggle/input").rglob("*"):
        if not d.is_dir():
            continue
        n = sum(1 for _ in d.glob("*.png"))
        if n > best_n:
            best, best_n = d, n
    return best, best_n


DATA_DIR, _n = find_data_dir()
print("Data:", DATA_DIR, f"({_n} png)")
assert DATA_DIR is not None and _n == 20000, "expected 20000 images"

BACKBONE      = "dinov2_vitl14"
SIZE          = 392     # multiple of 14
BATCH         = 12
EPOCHS        = 3
LR, HEAD_LR   = 1e-5, 1e-3
TEMPERATURE   = 0.05
TRAINABLE     = 4       # last N transformer blocks
STRENGTH      = 1.0
PCA_DIM       = 1536
SEED          = 0

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Enable Settings -> Accelerator -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}  "
      f"VRAM {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
torch.manual_seed(SEED); np.random.seed(SEED)

## Degradation

Two details are easy to get wrong and both matter. The downscale is anchored to
**absolute pixels**, because a real photograph has a fixed capture resolution
regardless of what size we feed the network. And the crop must not force a
square -- a real cropped query has its own aspect ratio, different from its
gallery original's, and squaring it here would hide exactly the mismatch the
model needs to tolerate.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Pale grey matches the padding observed in real query images (e.g. 16000.png).
PAD_FILL = 235

# Absolute pixel reference for the downscale, independent of model input size.
DOWNSCALE_REFERENCE = 224


class Downscale:
    """Shrink hard, then upsample back, destroying high-frequency detail.

    Blur alone does not model a low-resolution capture: it smooths without
    removing the information a larger sensor would have recorded.
    """

    def __init__(self, factor: float, reference: int = DOWNSCALE_REFERENCE):
        self.factor = factor
        self.reference = reference

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        target_long = max(16.0, self.reference / self.factor)
        scale = min(1.0, target_long / max(w, h))
        small = (max(8, int(w * scale)), max(8, int(h * scale)))
        return img.resize(small, Image.BILINEAR).resize((w, h), Image.BICUBIC)


class RandomCropAspect:
    """Crop a random sub-region, preserving a random (non-square) aspect ratio."""

    def __init__(self, scale=(0.55, 0.95), ratio=(0.7, 1.4)):
        self.scale, self.ratio = scale, ratio

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        area = w * h * np.random.uniform(*self.scale)
        ar = float(np.exp(np.random.uniform(np.log(self.ratio[0]), np.log(self.ratio[1]))))
        cw = min(w, max(8, int(round(np.sqrt(area * ar)))))
        ch = min(h, max(8, int(round(np.sqrt(area / ar)))))
        x = np.random.randint(0, w - cw + 1)
        y = np.random.randint(0, h - ch + 1)
        return img.crop((x, y, x + cw, y + ch))


def query_view(size: int, strength: float = 1.0) -> transforms.Compose:
    """The degraded view -- stands in for a visitor's photograph."""
    return transforms.Compose([
        transforms.RandomAffine(
            degrees=12 * strength,
            translate=(0.04 * strength, 0.04 * strength),
            scale=(1 - 0.15 * strength, 1 + 0.05 * strength),
            shear=6 * strength,
            fill=PAD_FILL,
        ),
        transforms.RandomPerspective(distortion_scale=0.22 * strength, p=0.85, fill=PAD_FILL),
        RandomCropAspect(),
        transforms.ColorJitter(
            brightness=0.3 * strength, contrast=0.3 * strength,
            saturation=0.25 * strength, hue=0.03 * strength,
        ),
        transforms.GaussianBlur(kernel_size=7, sigma=(0.4 * strength + 0.2, 2.4 * strength + 0.2)),
        Downscale(factor=1.0 + 2.5 * strength),
        transforms.Resize((size, size), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def gallery_view(size: int, jitter: bool = True) -> transforms.Compose:
    """The clean view -- stands in for the studio-captured gallery image.

    A little jitter is kept even on the anchor: the gallery side is not
    pixel-identical to what the model saw in training either, and a completely
    static anchor makes the contrastive task degenerately easy.
    """
    steps = [transforms.Resize((size, size), interpolation=transforms.InterpolationMode.BICUBIC)]
    if jitter:
        steps.append(transforms.ColorJitter(brightness=0.08, contrast=0.08))
    steps += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(steps)

## Paired dataset

Every image is a usable anchor. We are never told which of the 20,000 are
gallery images, queries or distractors, and it does not matter: "a degraded copy
of X retrieves X" is well-defined for any X.


In [ ]:
Image.MAX_IMAGE_PIXELS = None  # a few gallery scans are enormous


class PairDataset(Dataset):
    """Yields (anchor, positive, index) with the positive heavily degraded."""

    def __init__(self, paths: list[Path], size: int, strength: float = 1.0):
        self.paths = paths
        self.size = size
        self.anchor_tf = gallery_view(size)
        self.query_tf = query_view(size, strength)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int):
        path = self.paths[i]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:  # noqa: BLE001 -- a corrupt file must not kill training
            print(f"  ! unreadable {path.name}: {e}", flush=True)
            blank = torch.zeros(3, self.size, self.size)
            return blank, blank, i
        return self.anchor_tf(img), self.query_tf(img), i


class InferenceDataset(Dataset):
    """Clean, deterministic view for embedding the full corpus at submission time."""

    def __init__(self, paths: list[Path], size: int):
        self.paths = paths
        self.size = size
        self.tf = gallery_view(size, jitter=False)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception as e:  # noqa: BLE001
            print(f"  ! unreadable {self.paths[i].name}: {e}", flush=True)
            # Never drop a row -- the submission must carry all 20,000 images.
            return torch.zeros(3, self.size, self.size), i
        return self.tf(img), i

## Model

Only the last four blocks train -- fine-tuning all 300M parameters on 20,000
images would overwrite what DINOv2 learned from 142M and overfit. And the
projection head is discarded at inference: contrastive training warps the head's
output space to suit the loss, while the representation just beneath it
transfers better to retrieval.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def gem_pool(patch_tokens: torch.Tensor, p: float = 3.0, eps: float = 1e-6) -> torch.Tensor:
    """Generalised mean over patch tokens -- emphasises distinctive regions.

    Survives cropping better than the CLS token alone, because a crop removes
    some patches but leaves the informative ones dominating the mean.
    """
    return patch_tokens.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)


class RetrievalNet(nn.Module):
    def __init__(self, backbone_name: str = "dinov2_vitl14", trainable_blocks: int = 4,
                 proj_dim: int = 512):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        embed_dim = self.backbone.embed_dim

        for param in self.backbone.parameters():
            param.requires_grad = False
        for block in self.backbone.blocks[-trainable_blocks:]:
            for param in block.parameters():
                param.requires_grad = True
        for param in self.backbone.norm.parameters():
            param.requires_grad = True

        # CLS and GeM are concatenated, hence 2 * embed_dim.
        self.head = nn.Sequential(
            nn.Linear(2 * embed_dim, 2 * embed_dim),
            nn.GELU(),
            nn.Linear(2 * embed_dim, proj_dim),
        )

    def features(self, x: torch.Tensor) -> torch.Tensor:
        """Pooled backbone representation -- this is what gets submitted."""
        out = self.backbone.forward_features(x)
        cls = F.normalize(out["x_norm_clstoken"], dim=1)
        gem = F.normalize(gem_pool(out["x_norm_patchtokens"]), dim=1)
        return torch.cat([cls, gem], dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Projected embedding -- used only for the contrastive loss."""
        return F.normalize(self.head(self.features(x)), dim=1)

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

## Training

InfoNCE over in-batch negatives. Those negatives are the crux: the gallery holds
9,000 artwork distractors, so the model must separate *this* painting from other
paintings, not merely paintings from non-paintings.


In [ ]:
def info_nce(anchor, positive, temperature):
    logits = anchor @ positive.t() / temperature
    labels = torch.arange(len(anchor), device=anchor.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))


paths = sorted(DATA_DIR.glob("*.png"))
model = RetrievalNet(BACKBONE, TRAINABLE).to(device)
print(f"trainable {sum(p.numel() for p in model.trainable_parameters())/1e6:.1f}M / "
      f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

head_ids = {id(p) for p in model.head.parameters()}
optim = torch.optim.AdamW([
    {"params": [p for p in model.trainable_parameters() if id(p) not in head_ids], "lr": LR},
    {"params": list(model.head.parameters()), "lr": HEAD_LR},
], weight_decay=0.05)

loader = DataLoader(PairDataset(paths, SIZE, STRENGTH), batch_size=BATCH,
                    shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
steps_total = EPOCHS * len(loader)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optim, max_lr=[LR, HEAD_LR], total_steps=steps_total, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda")

t0, step = time.time(), 0
for epoch in range(EPOCHS):
    model.train()
    running = seen = 0
    for anchor, positive, _ in loader:
        anchor, positive = anchor.to(device, non_blocking=True), positive.to(device, non_blocking=True)
        optim.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = info_nce(model(anchor), model(positive), TEMPERATURE)
        scaler.scale(loss).backward()
        scaler.step(optim); scaler.update(); sched.step()

        running += loss.item() * len(anchor); seen += len(anchor); step += 1
        if step % 100 == 0:
            rate = step / (time.time() - t0)
            print(f"  epoch {epoch} step {step}/{steps_total}  loss {running/seen:.4f}  "
                  f"{rate:.2f} it/s  ETA {(steps_total-step)/rate/60:.0f} min", flush=True)

    torch.save({"model": model.state_dict(), "epoch": epoch},
               "/kaggle/working/finetuned.pt")
    print(f"epoch {epoch}: mean loss {running/seen:.4f}", flush=True)
print(f"trained in {(time.time()-t0)/60:.1f} min")

## Embed and submit

Post-processing is unchanged from the zero-shot pipeline: L2-normalise, whitened
PCA, L2-normalise. Whitening was the largest single lever in the zero-shot phase
-- with 9,000 artwork distractors the dominant variance directions encode "this
is a painting" and drown out what distinguishes one from another.


In [ ]:
def l2(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


loader = DataLoader(InferenceDataset(paths, SIZE), batch_size=BATCH * 2,
                    shuffle=False, num_workers=2, pin_memory=True)
feats, t0 = None, time.time()
model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    for imgs, idxs in loader:
        v = model.features(imgs.to(device, non_blocking=True)).float().cpu().numpy()
        if feats is None:
            feats = np.zeros((len(paths), v.shape[1]), dtype=np.float32)
        feats[idxs.numpy()] = v
print(f"embedded {feats.shape} in {(time.time()-t0)/60:.1f} min")

np.save("/kaggle/working/features_ft.npy", feats)   # keep for offline dimension tuning

x = l2(feats.astype(np.float64))
mu = x.mean(axis=0, keepdims=True)
_, s, vt = np.linalg.svd(x - mu, full_matrices=False)
dim = min(PCA_DIM, x.shape[1])
x = l2((x - mu) @ vt[:dim].T / (s[:dim] / np.sqrt(len(x) - 1) + 1e-8)).astype(np.float32)
print(f"PCA {feats.shape[1]} -> {dim}  explains {(s[:dim]**2).sum()/(s**2).sum():.1%}")

df = pd.DataFrame(x, columns=[f"feature_{i}" for i in range(x.shape[1])])
df.insert(0, "image_name", [p.name for p in paths])
df["ID"] = df["image_name"]
df.to_csv("/kaggle/working/submission.csv", index=False, float_format="%.6f")
print(f"rows={len(df)}  cols={df.shape[1]}  norms~{np.linalg.norm(x, axis=1).mean():.4f}")
df.iloc[:3, :5]